In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
import pickle

# Global dictionary to store model results
model_results = {}

# Load and Filter Data

In [2]:
# Load the dataset
data = pd.read_csv("bike_weather_data.csv")

# Data Cleaning
data['available_bikes'] = data['num_bikes_available']
data['temperature'] = (data['max_air_temperature_celsius'] + data['min_air_temperature_celsius']) / 2
data['humidity'] = (data['max_relative_humidity_percent'] + data['min_relative_humidity_percent']) / 2
data['day_of_week'] = pd.to_datetime(data['last_reported']).dt.dayofweek

features = ['station_id', 'temperature', 'humidity', 'hour', 'day_of_week']
target = 'available_bikes'

X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Data prepared. Training set size: {len(X_train)}, Test set size: {len(X_test)}")

Data prepared. Training set size: 209262, Test set size: 89684


# 1. Linear Regression

In [3]:
print("Training Linear Regression...")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

model_results['Linear Regression'] = {'model': lr_model, 'mae': mae, 'r2': r2}
print(f"Results -> MAE: {mae:.4f}, R²: {r2:.4f}")

Training Linear Regression...
Results -> MAE: 8.1424, R²: 0.0001


# 2. Decision Tree Regressor

In [4]:
print("Training Decision Tree...")
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

model_results['Decision Tree'] = {'model': dt_model, 'mae': mae, 'r2': r2}
print(f"Results -> MAE: {mae:.4f}, R²: {r2:.4f}")

Training Decision Tree...
Results -> MAE: 1.4807, R²: 0.8616


# 3. Random Forest Regressor

In [5]:
print("Training Random Forest...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

model_results['Random Forest'] = {'model': rf_model, 'mae': mae, 'r2': r2}
print(f"Results -> MAE: {mae:.4f}, R²: {r2:.4f}")

Training Random Forest...
Results -> MAE: 1.4673, R²: 0.9243


# Model Comparison and Selection

In [6]:
print("Comparing models based on MAE...")
for name, metrics in model_results.items():
    print(f"{name}: MAE = {metrics['mae']:.4f}, R² = {metrics['r2']:.4f}")

# Select the best model (lowest MAE)
best_model_name = min(model_results, key=lambda x: model_results[x]['mae'])
best_model = model_results[best_model_name]['model']

print(f"\nBest Model: {best_model_name}")

# Save the best model
joblib.dump(best_model, "bike_availability_model.joblib")
with open("bike_availability_model.pkl", "wb") as file:
    pickle.dump(best_model, file)

print(f"Best model saved as bike_availability_model.pkl")

Comparing models based on MAE...
Linear Regression: MAE = 8.1424, R² = 0.0001
Decision Tree: MAE = 1.4807, R² = 0.8616
Random Forest: MAE = 1.4673, R² = 0.9243

Best Model: Random Forest
Best model saved as bike_availability_model.pkl


# Model Prediction Demo

In [7]:
# Load the best saved model
with open("bike_availability_model.pkl", "rb") as file:
    final_model = pickle.load(file)

sample_input = pd.DataFrame({
    'station_id': [32],
    'temperature': [20],
    'humidity': [60],
    'hour': [9],
    'day_of_week': [2]
})

prediction = final_model.predict(sample_input)
print(f"Predicted available bikes: {prediction[0]:.2f}")

Predicted available bikes: 4.11
